In [ ]:
# DAPT + Adapter on CoastSent

This notebook is split into smaller cells for readability. It fine-tunes a frozen DAPT IndoBERT encoder with a lightweight residual adapter on CoastSent, evaluates on source and target test sets, and visualizes the adapted embeddings with t-SNE.

In [ ]:
# Imports
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments, DataCollatorWithPadding

sns.set_theme(style='whitegrid')

In [ ]:
# Reproducibility and device
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Configuration
DAPT_MODEL_DIR = './models/indobert_mlm_target_final'
OUTPUT_DIR = './models/indobert_coastsent_dapt_adapter'
BATCH_SIZE = 16
MAX_LENGTH = 128
EPOCHS = 3
LR = 1e-4
ADAPTER_DIM = 64
DROPOUT = 0.1
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs('outputs', exist_ok=True)

LABEL2ID = {'negative': 0, 'positive': 1}
ID2LABEL = {0: 'negative', 1: 'positive'}

print('Config loaded')
print('  DAPT model dir:', DAPT_MODEL_DIR)
print('  Output dir    :', OUTPUT_DIR)

In [ ]:
# Helpers
def resolve_text_column(df):
    for c in ['reviewContent', 'content', 'text', 'review']:
        if c in df.columns:
            return c
    raise ValueError('No text-like column found: ' + ', '.join(df.columns))


def normalize_labels(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .replace({
            'pos': 'positive',
            'neg': 'negative',
            'positif': 'positive',
            'negatif': 'negative',
            '1': 'positive',
            '0': 'negative',
            'true': 'positive',
            'false': 'negative',
        })
    )


def clean_df(df):
    df = df.copy()
    tc = resolve_text_column(df)
    df[tc] = df[tc].astype(str).str.strip()
    if 'label' not in df.columns:
        raise ValueError('Expected a label column in the dataset')
    df['label'] = normalize_labels(df['label'])
    df = df.dropna(subset=[tc, 'label'])
    df = df[df[tc] != '']
    df['label_id'] = df['label'].map(LABEL2ID)
    df = df.dropna(subset=['label_id'])
    df['label_id'] = df['label_id'].astype(int)
    return df, tc

# Load datasets
train_df = pd.read_csv('./datasets/coastsent_train.csv')
source_test_df = pd.read_csv('./datasets/coastsent_test.csv')
target_test_df = pd.read_csv('./datasets/lazada_test.csv')

train_df, train_text_col = clean_df(train_df)
source_test_df, source_text_col = clean_df(source_test_df)
target_test_df, target_text_col = clean_df(target_test_df)

print('Loaded shapes:')
print('  train :', train_df.shape)
print('  source:', source_test_df.shape)
print('  target:', target_test_df.shape)
print('Train label counts:', train_df['label_id'].value_counts().to_dict())

In [ ]:
# Tokenizer and adapter model
tokenizer = AutoTokenizer.from_pretrained(DAPT_MODEL_DIR)

class ResidualBottleneckAdapter(nn.Module):
    def __init__(self, hidden_size, adapter_dim=64, dropout=0.1):
        super().__init__()
        self.down = nn.Linear(hidden_size, adapter_dim)
        self.act = nn.GELU()
        self.up = nn.Linear(adapter_dim, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return x + self.dropout(self.up(self.act(self.down(x))))


class DAPTAdapterClassifier(nn.Module):
    def __init__(self, base_model_dir, adapter_dim=64, dropout=0.1, num_labels=2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_dir)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.adapter = ResidualBottleneckAdapter(hidden_size, adapter_dim=adapter_dim, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_labels),
        )

        for p in self.encoder.parameters():
            p.requires_grad = False

    def mean_pool(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

    def extract_features(self, input_ids, attention_mask, token_type_ids=None):
        feat = self.dropout(self.mean_pool(input_ids, attention_mask, token_type_ids=token_type_ids))
        return self.adapter(feat)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        feat = self.extract_features(input_ids, attention_mask, token_type_ids=token_type_ids)
        logits = self.classifier(feat)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return {'loss': loss, 'logits': logits} if loss is not None else {'logits': logits}


model = DAPTAdapterClassifier(
    base_model_dir=DAPT_MODEL_DIR,
    adapter_dim=ADAPTER_DIM,
    dropout=DROPOUT,
).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Model ready | trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Tokenization and datasets
def tokenize_fn(examples):
    enc = tokenizer(examples['text'], truncation=True, max_length=MAX_LENGTH)
    enc['labels'] = examples['labels']
    return enc

hf_train = Dataset.from_dict({
    'text': train_df[train_text_col].astype(str).tolist(),
    'labels': train_df['label_id'].tolist(),
})

split = hf_train.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
val_ds = split['test']

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=['text'])
val_tok = val_ds.map(tokenize_fn, batched=True, remove_columns=['text'])

source_eval_ds = Dataset.from_dict({
    'text': source_test_df[source_text_col].astype(str).tolist(),
    'labels': source_test_df['label_id'].tolist(),
}).map(tokenize_fn, batched=True, remove_columns=['text'])

target_eval_ds = Dataset.from_dict({
    'text': target_test_df[target_text_col].astype(str).tolist(),
    'labels': target_test_df['label_id'].tolist(),
}).map(tokenize_fn, batched=True, remove_columns=['text'])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print('Datasets tokenized:')
print('  train:', len(train_tok))
print('  val  :', len(val_tok))
print('  src  :', len(source_eval_ds))
print('  tgt  :', len(target_eval_ds))

In [ ]:
# Training setup and run
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Trainer ready')
train_result = trainer.train()
print(train_result.metrics)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved model to', OUTPUT_DIR)

In [ ]:
# Evaluation
def eval_on_dataset(ds, name='eval'):
    pred = trainer.predict(ds)
    preds = np.argmax(pred.predictions, axis=1)
    labels = pred.label_ids
    print(f'\nEvaluation on {name} - samples: {len(labels)}')
    print(classification_report(labels, preds, target_names=['negative', 'positive'], digits=4))
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

source_res = eval_on_dataset(source_eval_ds, name='coastSent test (source)')
target_res = eval_on_dataset(target_eval_ds, name='Lazada test (target)')

print('SUMMARY')
for k in ['accuracy', 'precision', 'recall', 'f1']:
    print(f'{k}: source={source_res[k]:.4f}  target={target_res[k]:.4f}  gap={source_res[k]-target_res[k]:+.4f}')

In [ ]:
# Visualization: encoder + adapter embeddings with t-SNE

def extract_embeddings_from_dataloader(model, dataloader, device):
    model.eval()
    Xs = []
    Ys = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch.get('token_type_ids')
            if isinstance(token_type_ids, torch.Tensor):
                token_type_ids = token_type_ids.to(device)
            else:
                token_type_ids = None

            feats = model.extract_features(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
            Xs.append(feats.cpu().numpy())

            labels = batch.get('labels')
            if isinstance(labels, torch.Tensor):
                Ys.append(labels.cpu().numpy())
            elif labels is not None:
                Ys.append(np.array(labels))
            else:
                Ys.append(np.full((feats.size(0),), -1, dtype=int))

    if not Xs:
        return np.zeros((0, model.encoder.config.hidden_size)), np.array([])

    return np.concatenate(Xs, axis=0), np.concatenate(Ys, axis=0)


def project_embeddings(X, method='tsne', n_components=2, random_state=42):
    if X.shape[0] == 0:
        return X
    if method == 'tsne':
        p = PCA(n_components=50, random_state=random_state).fit_transform(X) if X.shape[1] > 50 else X
        ts = TSNE(n_components=n_components, random_state=random_state, init='pca', perplexity=30)
        return ts.fit_transform(p)
    return PCA(n_components=n_components, random_state=random_state).fit_transform(X)


def plot_single_panel(proj, labels, domains, title, outpath=None, figsize=(8, 6)):
    palette = {
        ('source', 0): '#d62728',
        ('source', 1): '#1f77b4',
        ('target', 0): '#2ca02c',
        ('target', 1): '#9467bd',
    }
    display_names = {
        ('source', 0): 'source negative',
        ('source', 1): 'source positive',
        ('target', 0): 'target negative',
        ('target', 1): 'target positive',
    }

    fig, ax = plt.subplots(1, 1, figsize=figsize)
    domains = np.array(domains)
    xs, ys = proj[:, 0], proj[:, 1]

    for key in [('source', 0), ('source', 1), ('target', 0), ('target', 1)]:
        mask = (domains == key[0]) & (labels == key[1])
        if mask.sum() == 0:
            continue
        ax.scatter(xs[mask], ys[mask], s=20, alpha=0.75, c=palette[key], label=display_names[key])

    ax.set_title(title, fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=10, frameon=True)
    fig.tight_layout()
    if outpath:
        os.makedirs(os.path.dirname(outpath), exist_ok=True)
        fig.savefig(outpath, dpi=200)
    plt.show()

src_loader = DataLoader(source_eval_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)
tgt_loader = DataLoader(target_eval_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

Xs_src, Ys_src = extract_embeddings_from_dataloader(trainer.model, src_loader, device)
Xs_tgt, Ys_tgt = extract_embeddings_from_dataloader(trainer.model, tgt_loader, device)

X = np.concatenate([Xs_src, Xs_tgt], axis=0)
Y = np.concatenate([Ys_src, Ys_tgt], axis=0)
domains = np.array(['source'] * Xs_src.shape[0] + ['target'] * Xs_tgt.shape[0])
proj = project_embeddings(X, method='tsne', n_components=2, random_state=42)

outpath = 'outputs/tsne_dapt_adapter_coastsent_vs_lazada.png'
plot_single_panel(proj, Y, domains, title='t-SNE: DAPT + Adapter (coastSent source vs Lazada target)', outpath=outpath)
print(f'Saved t-SNE visualization to: {outpath}')